In [2]:
import os
from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

AttributeError: module 'matplotlib' has no attribute '_docstring'

### Load Dataset

In [ ]:
DATASET_PATH = "../data/Chest Cancer"

classes = [
    "Adenocarcinoma",
    "Large Cell Carcinoma",
    "Normal"
]

### Class Distribution

In [ ]:
class_counts = {}

for cls in classes:
    class_path = os.path.join(DATASET_PATH, cls)
    class_counts[cls] = len(os.listdir(class_path))

class_counts

In [ ]:
df_counts = pd.DataFrame(
    list(class_counts.items()),
    columns=["Class", "Count"]
)

df_counts

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(df_counts["Class"], df_counts["Count"])

plt.title("Chest Cancer - Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=20)

plt.show()

In [ ]:
image_data = []

for cls in classes:
    class_path = os.path.join(DATASET_PATH, cls)

    for file in os.listdir(class_path):
        file_path = os.path.join(class_path, file)

        try:
            with Image.open(file_path) as img:
                image_data.append({
                    "class": cls,
                    "filename": file,
                    "width": img.width,
                    "height": img.height,
                    "mode": img.mode,
                    "format": img.format
                })

        except Exception as e:
            print("Error:", file_path, e)

df_images = pd.DataFrame(image_data)

df_images.head()

In [ ]:
df_images[["width", "height"]].describe()

### Visualize Sample Images

Display random images from each cancer class to understand the visual characteristics of the dataset.

In [ ]:
fig, axes = plt.subplots(
    len(classes),
    5,
    figsize=(15, 12)
)

for row, cls in enumerate(classes):

    class_path = os.path.join(DATASET_PATH, cls)
    files = os.listdir(class_path)

    samples = np.random.choice(
        files,
        min(5, len(files)),
        replace=False
    )

    for col, file in enumerate(samples):

        file_path = os.path.join(class_path, file)
        img = Image.open(file_path)

        axes[row, col].imshow(img)
        axes[row, col].set_title(cls)
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()

### Check Image Formats and Modes

In [ ]:
print("Image formats:")
print(df_images["format"].value_counts())

print("\nImage modes:")
print(df_images["mode"].value_counts())

###  Check for Corrupted Images

In [ ]:
corrupted_images = []

for cls in classes:
    class_path = os.path.join(DATASET_PATH, cls)

    for file in os.listdir(class_path):
        file_path = os.path.join(class_path, file)

        try:
            with Image.open(file_path) as img:
                img.verify()

        except Exception:
            corrupted_images.append(file_path)

print("Corrupted images:", len(corrupted_images))

### Check for Duplicate Images

In [ ]:
import hashlib

hashes = {}

for cls in classes:
    class_path = os.path.join(DATASET_PATH, cls)

    for file in os.listdir(class_path):
        file_path = os.path.join(class_path, file)

        file_hash = hashlib.md5(
            open(file_path, "rb").read()
        ).hexdigest()

        hashes.setdefault(file_hash, []).append(file_path)

duplicates = {
    h: files
    for h, files in hashes.items()
    if len(files) > 1
}

print("Duplicate groups:", len(duplicates))

In [ ]:
duplicate_images = sum(
    len(files) for files in duplicates.values()
)

print("Duplicate groups:", len(duplicates))
print("Total duplicate images:", duplicate_images)

In [ ]:
duplicate_image_count = sum(
    len(files)
    for files in duplicates.values()
)

print("Duplicate groups:", len(duplicates))
print("Images involved in duplicate groups:", duplicate_image_count)

In [ ]:
cross_class_duplicates = []

for h, files in duplicates.items():

    file_classes = set()

    for file in files:
        for cls in classes:
            if os.path.join(DATASET_PATH, cls) in file:
                file_classes.add(cls)

    if len(file_classes) > 1:
        cross_class_duplicates.append((h, files, file_classes))

print("Duplicate groups across different classes:", len(cross_class_duplicates))

In [ ]:
for h, files, file_classes in cross_class_duplicates:
    print("Classes:", file_classes)
    print("Files:")
    
    for file in files:
        print(" ", file)

In [ ]:
print("Image Formats:")
print(df_images["format"].value_counts())

print("\nImage Modes:")
print(df_images["mode"].value_counts())

In [ ]:
df_images.groupby("class")["format"].value_counts()

In [ ]:
df_images.groupby("class")["mode"].value_counts()

In [ ]:
print("Corrupted images:", len(corrupted_images))

### Clean the cross-class duplicate
Before splitting, remove the conflicting duplicate.

In [ ]:
import os
import shutil

SOURCE_PATH = "../data/Chest Cancer"
CLEAN_PATH = "../data/Chest Cancer Clean"

classes = [
    "Adenocarcinoma",
    "Large Cell Carcinoma",
    "Normal"
]

for cls in classes:
    os.makedirs(os.path.join(CLEAN_PATH, cls), exist_ok=True)

In [ ]:
for cls in classes:

    source_class = os.path.join(SOURCE_PATH, cls)
    clean_class = os.path.join(CLEAN_PATH, cls)

    for file in os.listdir(source_class):

        if cls == "Large Cell Carcinoma" and file == "l4.png":
            continue

        source_file = os.path.join(source_class, file)
        clean_file = os.path.join(clean_class, file)

        shutil.copy2(source_file, clean_file)

### Create the cleaned dataset

In [ ]:
import os
import shutil
import hashlib

SOURCE_PATH = "../data/Chest Cancer"
CLEAN_PATH = "../data/Chest Cancer Clean"

classes = [
    "Adenocarcinoma",
    "Large Cell Carcinoma",
    "Normal"
]

for cls in classes:
    os.makedirs(os.path.join(CLEAN_PATH, cls), exist_ok=True)


# Find duplicate groups
hashes = {}

for cls in classes:

    class_path = os.path.join(SOURCE_PATH, cls)

    for file in os.listdir(class_path):

        file_path = os.path.join(class_path, file)

        with open(file_path, "rb") as f:
            file_hash = hashlib.md5(f.read()).hexdigest()

        hashes.setdefault(file_hash, []).append((cls, file))


# Copy images
kept = 0
removed = 0

for file_hash, files in hashes.items():

    duplicate_classes = set(cls for cls, file in files)

    # Remove cross-class duplicates
    if len(duplicate_classes) > 1:
        removed += len(files)
        continue

    # Keep one copy of same-class duplicates
    cls, file = files[0]

    source_file = os.path.join(SOURCE_PATH, cls, file)
    clean_file = os.path.join(CLEAN_PATH, cls, file)

    shutil.copy2(source_file, clean_file)

    kept += 1
    removed += len(files) - 1


print("Images kept:", kept)
print("Images removed:", removed)

In [ ]:
clean_counts = {}

for cls in classes:
    class_path = os.path.join(CLEAN_PATH, cls)
    clean_counts[cls] = len(os.listdir(class_path))

print(clean_counts)
print("Total cleaned images:", sum(clean_counts.values()))

In [ ]:
print("========== CHEST DATASET EDA SUMMARY ==========")

print("Total images: 982")

print("\nClass distribution:")
print("Adenocarcinoma: 333")
print("Large Cell Carcinoma: 177")
print("Normal: 215")
print("Squamous Cell Carcinoma: 257")

print("\nImage properties:")
print("Formats: PNG (966), JPEG (16)")
print("Modes: RGBA (905), RGB (74), L (3)")
print("Corrupted images: 0")

print("\nDuplicate analysis:")
print("Duplicate groups: 59")
print("Images involved in duplicate groups: 212")
print("Cross-class duplicate groups: 1")

print("\nCross-class duplicate:")
print("Large Cell Carcinoma/l4.png")
print("Squamous Cell Carcinoma/sq2.png")
print("These images were identical but had different labels.")

print("\nDataset observations:")
print("- The dataset is class imbalanced.")
print("- Image formats and modes are inconsistent.")
print("- Duplicate images are present.")
print("- One cross-class duplicate indicates a potential label conflict.")

print("\n===============================================")